# 第 1 周末练习 —— 技术问答解释器（Ollama / Llama）

## 练习目标（理念）

为了展示你对 **OpenAI 兼容 API** 以及本地 **Ollama** 的熟悉程度，请构建一个小工具：

- **输入**：一个技术问题（例如「这段 Python 代码在干什么？」）
- **输出**：清晰、严谨的解释（面向工程经理的语气）
- **额外要求**：用**流式（streaming）**一边生成一边刷新 Markdown 显示

这是你在课程期间自己也能天天用的工具：遇到看不懂的代码，丢进来问模型。

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions API | `chat.completions.create(...)` |
| `messages`（system / user） | system 定「怎么答」，user 放具体问题 |
| 流式输出 `stream=True` | 逐块拼接，用 `update_display` 刷新 |
| Ollama + OpenAI SDK | 通过 `base_url` 指向 Ollama，仍用 `OpenAI` 客户端 |

## 怎么跑

1. 从上到下依次运行每个单元格（Shift+Enter）
2. 准备好 `.env`：需要 `OLLAMA_API_KEY` 与 `OLLAMA_BASE_URL`（例如 OpenAI 兼容地址）
3. 确保本机 Ollama 已拉取 `llama3.2`，并在提问单元格改写问题字符串


In [ ]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 导入标准库 os：读环境变量（Environment Variables），例如 API Key、Base URL
import os
# 从 dotenv 导入 load_dotenv：把 .env 文件里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 openai 导入 OpenAI 客户端类：这里会用它连接 Ollama 的 OpenAI 兼容接口
from openai import OpenAI
# 从 IPython.display 导入展示工具：Markdown 渲染、display 与流式刷新 update_display
from IPython.display import Markdown, display, update_display


In [ ]:
# ========== 常量：模型名字集中写在一处，后面只改这里 ==========

# OpenAI 云端小模型名（本笔记本后面主要走 Llama；保留常量便于扩展对比）
MODEL_GPT = 'gpt-4o-mini'
# 本地 Ollama 模型名：需事先 ollama pull llama3.2；字符串必须和本机已安装的模型名一致
MODEL_LLAMA = 'llama3.2'


In [ ]:
# ========== 环境变量 + 客户端：用 OpenAI SDK 对接 Ollama ==========

# 加载 .env：override=True 表示用文件里的值覆盖已有环境变量
load_dotenv(override=True)

# 从环境变量读取 Ollama 侧使用的 API Key（名字以本练习为准：OLLAMA_API_KEY）
api_key = os.getenv("OLLAMA_API_KEY")

# 从环境变量读取 OpenAI 兼容的 Base URL（例如指向本地或代理的 /v1）
base_url = os.getenv("OLLAMA_BASE_URL")

# 启动前自检：缺密钥或缺地址时打印提示（错误文案保持英文，便于对照排错笔记本）
if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not base_url:
    print("An base url  was found, please check see troubleshooting notebook")
else:
    print("API key and base url found and looks good so far!")

# 创建客户端：base_url 指向 Ollama（或兼容网关），api_key 一并传入
llamaAi = OpenAI(base_url=base_url, api_key=api_key)


In [ ]:
# ========== System Prompt：规定模型「以什么身份、怎么回答」 ==========

# system_prompt 发给模型的指令字符串：保留英文，改译会改变回答风格/行为
# 理念：角色 = 帮工程经理答疑的助手；不确定时要说出来，避免误导
system_prompt = """ 
You're a helpful assistant who answers to an engineering manager
You have knowledge across various technical subjects. 
You aim to give accurate answers to questions, and if you're not confident in the answer to a question, you always tell state that, so that you do not mislead your boss.

"""


In [ ]:
# ========== 核心函数 ask：流式调用 Llama，并在笔记本里实时刷新 Markdown ==========

def ask(question):
    # 调用 Chat Completions：model 用本地 Llama；messages 含 system + user；stream=True 开启流式
    stream = llamaAi.chat.completions.create(
        model= MODEL_LLAMA,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ],
        stream=True
    )
    # 累加器：把每个 chunk 里的增量文本拼成完整回答
    response = ''

    # 先放一个空的 Markdown 占位，拿到 display_id，方便后面原地更新（打字机效果）
    display_handle = display(Markdown(""), display_id=True)

    # 遍历流式响应：每个 chunk 可能带一小段 delta.content
    for chunk in stream:
        # or ''：若该 chunk 没有 content（例如结束标记），当作空串，避免 TypeError
        response += chunk.choices[0].delta.content or ''
        # 用同一 display_id 刷新整段 Markdown，实现边生成边显示
        update_display(Markdown(response), display_id=display_handle.display_id)


In [ ]:
# ========== 提问并调用：改三引号里的问题就能问新内容 ==========

# 发给模型的 user 内容保持英文（可运行 / 影响回答的字符串不翻译）
# 练习建议：换成你自己今天看不懂的一行代码，再跑本格对比解释质量
ask(""" 
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
""")
